# **Cancellation Prediction**

## Objectives

* Extend the preprocessing pipeline with transformations/encoding appropriate for tree-based classification
* Compare candidate classification algorithms using cross-validated recall to identify suitable models
* Conduct hyperparameter tuning on the leading candidate models
* Fit the final classification pipeline (preprocessing + selected model) on the full training set

## Inputs

* Train and test datasets from "outputs/ml_pipeline/preprocessing/" 
* Preprocessing pipeline "outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl"

## Outputs

* Classification preprocessing pipeline saved to "outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl"
* Classification modelling pipeline saved to "outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl"


## Additional Comments

* Model evaluation was moved to a separate notebook for readability; this notebook covers algorithm selection and hyperparameter tuning only.
* n_jobs was initially set to -1, but hardware limitations caused issues with this so it was changed to 1.


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

---

## Load Data

In [ ]:
import pandas as pd

X_train = pd.read_csv("outputs/ml_pipeline/preprocessing/X_train.csv")
X_test = pd.read_csv("outputs/ml_pipeline/preprocessing/X_test.csv")
y_train = pd.read_csv("outputs/ml_pipeline/preprocessing/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/ml_pipeline/preprocessing/y_test.csv").squeeze()

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

---

# Add Model Specific Preprocessing

**Classification Pipeline Actions**

| **Feature** | **Prediction model actions** | **Experimental prediction model alternatives** |
| --- | --- | --- |
| is_canceled | Target only | SMOTE |
| lead_time | | Log/skew transformation; binning |
| arrival_date_month | One-hot | Cyclical encoding |
| arrival_date_week_number | Exclude | Cyclical encoding |
|stays_in_weekend_nights||binning|
|stays_in_week_nights||binning|
|adults||binning|
|children||binary, binning|
|babies||binary, binning|
| country | Ordinal encoding | Frequency encoding; target encoding |
| previous_cancellations | | Binary |
| previous_bookings_not_canceled | | Log/skew transformation; binning |
| booking_changes | | Binary |
| agent | | Frequency encoding; target encoding; exclude |
| days_in_waiting_list | | Binary; log/skew transformation |
| adr | | Log/skew transformation; binning |
| required_car_parking_spaces | | Binary |
| total_of_special_requests | | Binning; binary |


In [ ]:
drop = ["arrival_date_week_number"]
ordinal = ["country"]
one_hot = ["arrival_date_month"]

* Load and test the preprocessing pipeline

In [ ]:
data = X_train.copy()
data.head(3)

In [ ]:
import joblib
from utils.custom_transformers import undefined_meal

preprocessing_pipeline = joblib.load("outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")
pipeline_step1 = preprocessing_pipeline.fit_transform(data)
pipeline_step1.shape


In [ ]:
pipeline_step1.head()

In [ ]:
pipeline_step1["country"].isnull().sum()

* Drop `arrival_date_week_number` 

In [ ]:
from feature_engine.selection import DropFeatures

drop_transformer = DropFeatures(features_to_drop=drop)
pipeline_step2 = drop_transformer.fit_transform(pipeline_step1)
pipeline_step2.shape

* Encode `country`

In [ ]:
from feature_engine.encoding import OrdinalEncoder

encoder = OrdinalEncoder(encoding_method="arbitrary", variables=ordinal)
pipeline_step3 = encoder.fit_transform(pipeline_step2)
pipeline_step3["country"].head(3)

In [ ]:
pipeline_step3["country"].isnull().sum()

* One-hot encode `arrival_date_month`

In [ ]:
from feature_engine.encoding import OneHotEncoder

encoder = OneHotEncoder(variables=one_hot)
pipeline_step4 = encoder.fit_transform(pipeline_step3)
pipeline_step4.shape

* Build the model specific preprocessing pipeline and test

In [ ]:
from feature_engine.encoding import RareLabelEncoder
from sklearn.pipeline import Pipeline

def classification_preprocessing_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", preprocessing_pipeline),
        ("DropFeatures", DropFeatures(features_to_drop=drop)),
        ("RareLabelEncoder", RareLabelEncoder(tol=0.01, variables=ordinal)),
        ("OrdinalEncoder", OrdinalEncoder(encoding_method="arbitrary", variables=ordinal, ignore_format=True)),
        ("OneHotEncoder", OneHotEncoder(variables=one_hot))        
    ])

    return pipeline_base

* Unplanned RareLabelEncoder added because the OrdinalEncoder added NaN values on the unseen test set

In [ ]:
test_df = X_train.copy()
classification_model_preprocessing_pipeline = classification_preprocessing_pipeline()
pipeline_test = classification_model_preprocessing_pipeline.fit_transform(test_df)
pipeline_test.shape

In [ ]:
classification_model_preprocessing_pipeline.named_steps

In [ ]:
pipeline_test["country"].isnull().sum()

* Test that there are no remaining NaN values in `country`

In [ ]:
encoding_error_test = X_test.copy()
error_test = classification_model_preprocessing_pipeline.transform(encoding_error_test)
error_test["country"].isnull().sum()

---

## Classification modelling pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler

def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("Scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline_base

* Model selection

In [ ]:
# Code adapted from the Churnometer walkthrough

from sklearn.model_selection import GridSearchCV
import numpy as np


class ModelComparison:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = classification_pipeline(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

* Use standard hyperparameters to find the most suitable model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

models_search = {
    "LogisticRegression": LogisticRegression(random_state=0, max_iter=500),  # Default value of 100 was insufficient 
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=0),
}

params_search = {
    "LogisticRegression": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "XGBClassifier": {},
    "GradientBoostingClassifier": {},
}

In [ ]:
from sklearn.metrics import make_scorer, recall_score
search = ModelComparison(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

* Multiple algorithms were evaluated using identical processing and 5-fold ross-validation
* Recall was used as the primary optimisation metric because identifying cancellations is the main business requirement
* LogisticRegression was tested to evaluate linear relationships, and to provide a baseline. The task is binary classification so there was little expectation of a strong performance and this was backed up by the results.
* Tree-based methods performed substantially better with both RandomForest and XGBoost achieving 0.802 which meets the target recall of 0.8 set out in the business understanding. These will be carried forward for tuning and evaluation. There is nothing to separate the two models at this stage in terms of mean_score, their min/max scores are also comparable at RF: 0.797/0.807 and XGB: 0.794/0.813
* DecisionTree was very close at 0.798 and with a max_score of 0.802 could be considered as alternative if later testing of the primary 2 models doesn't provide the desired results.
* std_score for all models tested was less than 0.01 suggesting stability across the folds in all models.
* GradientBoosting and LogisticRegression are dropped at this stage since neither achieved the target recall value with their max_scores (0.733 and 0.633 repectively)
 

---

## Hyperparameter tuning 

* remove scaling from the pipeline since LogisticRegression discounted

In [ ]:
def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", model)
    ])

    return pipeline_base

* Create function to compare hyperparameter performance

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

def parameter_comparison(model, param, values):

    results = []

    for value in values:

        current_model = clone(model)
        current_model.set_params(**{param: value})

        scores = cross_val_score(
            classification_pipeline(current_model),
            X_train,
            y_train.values.ravel(),
            scoring="recall",
            cv=5,
            verbose=1
        )

        results.append({
            "parameter": param,
            "value": value,
            "recall": scores.mean()
        })

    return pd.DataFrame(results)

**RandomForestClassification**

In [ ]:
random_forest = RandomForestClassifier(random_state=0)

In [ ]:
max_depth_results = parameter_comparison(
    model=random_forest,
    param="max_depth",
    values=[None, 10, 20]
)
max_depth_results

In [ ]:
n_estimators_results = parameter_comparison(
    model=random_forest,
    param="n_estimators",
    values=[100, 300, 500]
)
n_estimators_results

In [ ]:
min_samples_leaf_results = parameter_comparison(
    model=random_forest, 
    param="min_samples_leaf", 
    values=[1, 2, 4])
min_samples_leaf_results

* None of the options tested made significant improvements to recall with the highest gain being 0.002 for n_estimators=5 and this added significantly more training time than the default
* Default settings are already optimal for this model.

**XGBClassifier**

In [ ]:
xgb = XGBClassifier(random_state=0)

In [ ]:
xgb_max_depth_results = parameter_comparison(
    model=xgb, 
    param="max_depth", 
    values=[3, 5, 7, 10])
xgb_max_depth_results

In [ ]:
xgb_n_estimators_results = parameter_comparison(
    model=xgb, 
    param="n_estimators", 
    values=[100, 200, 300, 500])
xgb_n_estimators_results

In [ ]:
xgb_learning_rate_reults = parameter_comparison(
    model=xgb,
    param="learning_rate",
    values = [0.1, 0.2, 0.3, 0.4]
)
xgb_learning_rate_reults

In [ ]:
xgb_subsample_results = parameter_comparison(
    model=xgb,
    param="subsample",
    values=[0.6, 0.8, 1.0]
)
xgb_subsample_results

In [ ]:
xgb_colsample_bytree_results = parameter_comparison(
    model=xgb,
    param="colsample_bytree",
    values=[0.6, 0.8, 1.0]
)
xgb_colsample_bytree_results

* max_depth of 10 produced the largest improvement in recall (0.802 to 0.824), this value will be carried forward.
* All other changes produced little to no effect, so default values will be retained

In [ ]:
models_search = {
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
}

params_search = {
    "RandomForestClassifier": {},
    "XGBClassifier": {"model__max_depth": [10], "model__n_estimators": [500]},
}

In [ ]:
optimised_models = ModelComparison(models=models_search, params=params_search)
optimised_models.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)

In [ ]:
grid_search_summary, grid_search_pipelines = optimised_models.score_summary(sort_by='mean_score')
grid_search_summary

* From this result I will be taking XGBClassifier forward with max_depth=10 and n_estimators=500 for further testing.
* It is the faster of the 2 models, so even if the hyperparameters later need to be removed in case of overfitting, it is still the better choice

In [ ]:
def classification_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", XGBClassifier(max_depth=10, n_estimators=500, random_state=0))
    ])

    return pipeline_base

In [ ]:
X = X_train.copy()
y = y_train.copy()

Xtest = X_test.copy()
ytest = y_test.copy()

classification_model_pipeline = classification_pipeline()
classification_model_pipeline.fit(X, y)

---

## Save Files

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/cancel_predict/v1')
except Exception as e:
  print(e)


* Save prediction preprocessing pipeline

In [ ]:
import joblib

joblib.dump(value=classification_model_preprocessing_pipeline, filename="outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl")

* Save prediction pipeline

In [ ]:
joblib.dump(value=classification_model_pipeline, filename="outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl")